In [ ]:
import os
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# Testing: Persona Mimic Evaluation

Evaluate mimic responses against reference answers using content and style metrics, and summarize dataset-level performance.


In [ ]:

# Setup: imports
from __future__ import annotations
import os, sys, json, re, math
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from openai import OpenAI

# Ensure UTF-8 stdout
if hasattr(sys, "stdout"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

# NLTK resources (tokenizer)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)


In [ ]:

# Configuration
INPUT_PATH: Path = Path("/Users/nishchayjaiswal/Documents/Development/Verdict POC/V2/data/processed/mimic_outputs.json")
OUTPUT_PATH: Path = Path("./metric_results.json")

# NLI model (MNLI-compatible DeBERTa)
NLI_MODEL_NAME: str = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# Sentence embedding model
EMBEDDING_MODEL_NAME: str = "sentence-transformers/all-MiniLM-L6-v2"

# LLM style judge (OpenAI Responses API model)
LLM_STYLE_MODEL: str = "gpt-5"

# Control
MAX_EXAMPLES: Optional[int] = None  # None = all
RUN_LLM_STYLE: bool = True  # set False to skip OpenAI style judging

print(f"Input: {INPUT_PATH} Output: {OUTPUT_PATH} NLI: {NLI_MODEL_NAME} Embedder: {EMBEDDING_MODEL_NAME} Style model: {LLM_STYLE_MODEL} Max examples: {MAX_EXAMPLES} Run LLM style: {RUN_LLM_STYLE}")


In [ ]:

# API key and OpenAI client (only needed if RUN_LLM_STYLE is True)
# removed hardcoded env assignment
client = None
if RUN_LLM_STYLE:
    if not os.getenv("OPENAI_API_KEY"):
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (hidden): ")
        if not os.environ["OPENAI_API_KEY"]:
            raise RuntimeError("OPENAI_API_KEY not provided, but RUN_LLM_STYLE=True.")
    client = OpenAI()
    print("OpenAI client initialized.")
else:
    print("RUN_LLM_STYLE is False: skipping OpenAI client init.")


In [ ]:

# Helpers: load, metrics, models

def load_records(path: Path) -> List[Dict[str, Any]]:
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Input JSON must be a list of records.")
    out: List[Dict[str, Any]] = []
    for i, rec in enumerate(data):
        if not isinstance(rec, dict):
            continue
        question = rec.get('question')
        verdict = rec.get('verdict')
        beliefs = rec.get('beliefs', [])
        mimic = rec.get('mimic_response')
        raw = rec.get('raw_answer')
        if not isinstance(verdict, str) or not verdict.strip():
            raise ValueError(f"Record {i}: missing or invalid 'verdict'.")
        if not isinstance(mimic, str) or not mimic.strip():
            raise ValueError(f"Record {i}: missing or invalid 'mimic_response'.")
        if not isinstance(raw, str) or not raw.strip():
            raise ValueError(f"Record {i}: missing or invalid 'raw_answer'.")
        if beliefs is None:
            beliefs = []
        if isinstance(beliefs, str):
            beliefs = [beliefs]
        if not isinstance(beliefs, list):
            raise ValueError(f"Record {i}: 'beliefs' must be a list or string.")
        out_rec = dict(rec)
        out_rec['beliefs'] = [str(b) for b in beliefs]
        out.append(out_rec)
    if not out:
        raise ValueError("No valid records found.")
    return out


def load_nli_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    return tokenizer, model


def _entailment_index_from_config(model) -> Optional[int]:
    # Try to find the index of the entailment label in model config
    try:
        id2label = getattr(model.config, 'id2label', None)
        if isinstance(id2label, dict):
            for idx, name in id2label.items():
                if isinstance(name, str) and 'entail' in name.lower():
                    return int(idx)
    except Exception:
        pass
    return None


def entailment_probability(premise: str, hypothesis: str, tokenizer, model, label_map: Optional[Dict[int, str]] = None) -> float:
    inputs = tokenizer(premise, hypothesis, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    # Determine entailment index
    ent_idx = _entailment_index_from_config(model)
    if ent_idx is None and label_map:
        for idx, name in label_map.items():
            if 'entail' in str(name).lower():
                ent_idx = idx
                break
    if ent_idx is None:
        # Fallback heuristics for common MNLI heads
        # Try label order [entailment, neutral, contradiction]
        # Many HF MNLI models: 0=entailment, 1=neutral, 2=contradiction
        if logits.shape[-1] == 3:
            # Heuristic: choose the label with name matching if available
            ent_idx = 0
        else:
            ent_idx = int(np.argmax(probs))
    return float(probs[ent_idx])


def compute_avg_entailment_score(record: Dict[str, Any], tokenizer, model) -> Dict[str, Any]:
    premise = record.get('mimic_response', '')
    verdict = record.get('verdict', '')
    beliefs: List[str] = record.get('beliefs', []) or []
    scores = []
    verdict_score = entailment_probability(premise, verdict, tokenizer, model)
    scores.append(verdict_score)
    belief_scores = []
    for b in beliefs:
        s = entailment_probability(premise, b, tokenizer, model)
        belief_scores.append(s)
        scores.append(s)
    avg_score = float(np.mean(scores)) if scores else 0.0
    belief_mean = float(np.mean(belief_scores)) if belief_scores else None
    return {
        'avg_entailment_score': avg_score,
        'verdict_entailment_score': verdict_score,
        'belief_entailment_scores': belief_scores,
        'belief_entailment_mean': belief_mean,
    }


def _tokenize_for_bleu(text: str) -> List[str]:
    try:
        from nltk.tokenize import word_tokenize
        return word_tokenize(text)
    except Exception:
        return text.split()


def compute_bleu(reference: str, candidate: str) -> float:
    ref_tokens = _tokenize_for_bleu(reference)
    cand_tokens = _tokenize_for_bleu(candidate)
    smoothie = SmoothingFunction().method1
    try:
        return float(sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie))
    except ZeroDivisionError:
        return 0.0


def load_embedding_model(model_name: str):
    return SentenceTransformer(model_name)


def compute_embedding_similarity(reference: str, candidate: str, embedder) -> float:
    v = embedder.encode([reference, candidate], convert_to_numpy=True, normalize_embeddings=True)
    ref_vec, cand_vec = v[0], v[1]
    sim = float(np.dot(ref_vec, cand_vec))
    return sim


from typing import Any, List, Optional, Tuple
import json
from openai import OpenAI


def get_output_text(response: Any) -> str:
    txt = getattr(response, "output_text", None)
    if isinstance(txt, str) and txt.strip():
        return txt

    try:
        parts: List[str] = []
        for item in getattr(response, "output", []) or []:
            for content in getattr(item, "content", []) or []:
                if getattr(content, "type", "") == "output_text":
                    parts.append(getattr(content, "text", ""))
        if parts:
            return "\n".join([p for p in parts if p])
    except Exception:
        pass

    raise ValueError("Could not extract output text from response.")


def judge_style_with_llm(
    raw_answer: str,
    mimic_response: str,
    client: Optional[OpenAI],
    model_name: str,
) -> Tuple[Optional[float], Optional[str]]:
    if client is None:
        return None, None

    system_prompt = """
You are evaluating whether two responses sound like they were written or spoken by the same person.

Focus only on style, not content correctness.

Judge based on:
- sentence flow
- cadence
- level of polish
- repetition patterns
- directness
- phrasing style
- how spoken vs written it feels
- DO NOT judge based on punctuation

You must return valid JSON only with this schema:
{
  "style_score": <number from 1 to 10>,
  "reason": "<short explanation>"
}

Scoring:
1 = completely different style
10 = extremely similar style

Be strict. Do not inflate scores.
""".strip()

    user_prompt = f"""
Compare these two responses.

Reference response:
{raw_answer}

Generated response:
{mimic_response}

Return the JSON now.
""".strip()

    resp = client.responses.create(
        model=model_name,
        instructions=system_prompt,
        input=user_prompt,
        text={"format": {"type": "json_object"}},
    )

    out_text = get_output_text(resp)

    try:
        obj = json.loads(out_text)
        score = float(obj.get("style_score")) if obj.get("style_score") is not None else None
        reason = obj.get("reason")
        return score, reason
    except Exception:
        start = out_text.find("{")
        end = out_text.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                obj = json.loads(out_text[start:end + 1])
                score = float(obj.get("style_score")) if obj.get("style_score") is not None else None
                reason = obj.get("reason")
                return score, reason
            except Exception:
                pass
        raise


def evaluate_record(record: Dict[str, Any], tokenizer, nli_model, embedder, client: Optional[OpenAI], llm_model_name: str) -> Dict[str, Any]:
    # Content: entailment
    entail = compute_avg_entailment_score(record, tokenizer, nli_model)
    # Style: BLEU
    bleu = compute_bleu(record.get('raw_answer', ''), record.get('mimic_response', ''))
    # Style: embedding similarity
    emb_sim = compute_embedding_similarity(record.get('raw_answer', ''), record.get('mimic_response', ''), embedder)
    # LLM style judge (optional)
    if client is not None:
        style_score, style_reason = judge_style_with_llm(record.get('raw_answer', ''), record.get('mimic_response', ''), client, llm_model_name)
    else:
        style_score, style_reason = None, None
    result = {
        'id': record.get('id'),
        'question': record.get('question'),
        **entail,
        'bleu_score': bleu,
        'embedding_similarity': emb_sim,
        'llm_style_score': style_score,
        'llm_style_reason': style_reason,
    }
    return result


def save_results(results: List[Dict[str, Any]], summary: Dict[str, Any], output_path: Path) -> None:
    output = {'summary': summary, 'results': results}
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)


In [ ]:

# Load data
try:
    records = load_records(INPUT_PATH)
    if isinstance(MAX_EXAMPLES, int) and MAX_EXAMPLES > 0:
        records = records[:MAX_EXAMPLES]
    print(f"Loaded {len(records)} records.")
    # Preview first record minimal fields
    preview = {k: records[0].get(k) for k in ['id','question','verdict','beliefs','mimic_response','raw_answer']}
    print(json.dumps(preview, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"Failed to load data: {e}")
    raise


In [ ]:

# Initialize models
try:
    nli_tokenizer, nli_model = load_nli_model(NLI_MODEL_NAME)
    print("Loaded NLI model.")
except Exception as e:
    print(f"Failed to load NLI model: {e}")
    raise

try:
    embedder = load_embedding_model(EMBEDDING_MODEL_NAME)
    print("Loaded sentence embedding model.")
except Exception as e:
    print(f"Failed to load embedding model: {e}")
    raise


In [ ]:

# Compute per-record metrics
results: List[Dict[str, Any]] = []
for idx, rec in enumerate(records, 1):
    try:
        res = evaluate_record(rec, nli_tokenizer, nli_model, embedder, client if RUN_LLM_STYLE else None, LLM_STYLE_MODEL)
        results.append(res)
        if idx == 1:
            print("First result preview:", json.dumps(res, indent=2, ensure_ascii=False))
    except Exception as e:
        print(f"Evaluation failed at record {idx}: {e}")
        raise

print(f"Computed metrics for {len(results)} records.")


In [ ]:

# Results DataFrame
cols = ['id','avg_entailment_score','bleu_score','embedding_similarity','llm_style_score', 'llm_style_reason']
df = pd.DataFrame(results)
# Ensure required columns exist
for c in cols:
    if c not in df.columns:
        df[c] = None

display(df[cols].head(20))


In [ ]:

# Dataset summary
summary = {
    'num_records': len(results),
    'avg_entailment_score_mean': float(np.mean([r['avg_entailment_score'] for r in results])) if results else 0.0,
    'bleu_score_mean': float(np.mean([r['bleu_score'] for r in results])) if results else 0.0,
    'embedding_similarity_mean': float(np.mean([r['embedding_similarity'] for r in results])) if results else 0.0,
    'llm_style_score_mean': float(np.mean([r['llm_style_score'] for r in results if r.get('llm_style_score') is not None])) if any(r.get('llm_style_score') is not None for r in results) else None,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:

# Save results to JSON
try:
    save_results(results, summary, OUTPUT_PATH)
    print(f"Saved results to: {OUTPUT_PATH.resolve()}")
except Exception as e:
    print(f"Failed to save results: {e}")
    raise
